In [1]:
import os
import re
import json
import pickle

import tqdm
import numpy as np
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
SEED = 23
TEMPERATURE = 0.1

In [4]:
MODEL = "google/gemini-2.5-flash"

In [5]:
output_dir_name = MODEL.split("/")[1]

In [6]:
df = pd.read_excel("data_clean.xlsx", header=0)

In [7]:
df.head(10)

,title_rus,title_eng,annotation,description
0,Исследование приоритетов и механизмов реализац...,Study of Priorities and Mechanisms for Impleme...,Работа международных фондов (доноров) должна п...,"Согласно определению международных фондов, про..."
1,Антрополе - научно-популярный видео-подкаст о ...,Anthropole is a Popular Science Video Podcast ...,"\tАнтрополе - научно-популярный проект, в рамк...","Социальное знание близко и интересно обществу,..."
2,"Разработка, создание и ведение сайта, посвящен...","Design, Development and Implementation of a We...",Художественное образование и творчество художн...,Тема обучения арабских художников в художестве...
3,Перевод с английского языка коллективной моног...,Translation from English of the collective mon...,"Коллективная монография, авторы которой являют...","Коллективная монография, авторы которой являют..."
4,Сеть военно-политических союзов в Евразии: баз...,Network of Military in Eurasia: a Database,Проект посвящен изучению сети военно-политичес...,Проект посвящен анализу истории существования ...
5,Дизайн учебных материалов (презентаций),Instructional Materials Design (PPT Presentati...,Участникам проекта предлагается оформить две у...,Необходимо оформить две учебные презентации в ...
6,ОРЗ Изучении практик многоязычного обучения в ...,Studying the Practices of Polylingual Educatio...,Участие в нашем проекте будет интересен для те...,Преподавание родного языка и родной литературы...
7,Привязанность к месту и психосоциальное здоров...,Place attachment and psychosocial health: a me...,Привязанность к месту (place attachment) — это...,Привязанность к месту (place attachment) — сра...
8,Подготовка управленческих решений на корпорати...,Preparation of Managerial Decisions at the Co...,Для осуществления управленческих функций необх...,Сложности подготовки управленческих решений на...
9,"Дизайн лонгрида ""Корея в Москве и Санкт-Петерб...","Design of a Longread ""Korea in Moscow and St. ...",Подготовка привлекательного визуального отобра...,Сейчас завершается финальная часть проекта по ...


In [ ]:
with open(f"{output_dir_name}/artificial_profiles.json", "r") as f:
    profiles = json.load(f)

In [ ]:
with open(f"gemma-4-E2B-it/titles_with_tags_dict.pkl", "rb") as f:
    titles_with_tags_dict = pickle.load(f)

In [ ]:
data = []

for title in titles_with_tags_dict:
    for tag in titles_with_tags_dict[title]:
        data.append((title, tag))

titles_df = pd.DataFrame(data=data, columns=["title", "tag"])

In [11]:
titles_df.head()

,title,tag
0,Исследование приоритетов и механизмов реализац...,international_relations
1,Исследование приоритетов и механизмов реализац...,political_science
2,Исследование приоритетов и механизмов реализац...,economics
3,Исследование приоритетов и механизмов реализац...,development_studies
4,Исследование приоритетов и механизмов реализац...,BRICS


In [12]:
client = OpenAI(
    api_key=os.environ.get("KODIK_API_KEY"),
    base_url="https://api.kodikrouter.ru/v1",
)

In [13]:
results = {}

In [14]:
for profile, description in tqdm.tqdm(list(profiles.items())):

    SYSTEM_PROMPT = f"""
    You are a {profile}.
    Yours bio: '{description['bio']}'
    Yours tags: '{description['tags']}'
    Evaluate, how much you would like proposed student project based on its title in russian and tags. Rate it on a scale [1, 2, 3, 4, 5], where the higher the better.
    Return stricly JSON.
    For example.
    """
    SYSTEM_PROMPT += """
    Input.
    ```json
    {'title': 'Исследование методов решения задачи линейной регрессии', 'tags': ['machine_learning', 'algorithms']}
    ```
    """
    SYSTEM_PROMPT += """
    Output.
    ```json
    {'score': 5}
    ```
    """

    messages = [
        (
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": json.dumps(
                    {
                        "title": row[1]["title"],
                        "tags": titles_with_tags[row[1]["title"]],
                    },
                    ensure_ascii=False,
                ),
            },
        )
        for row in titles_df[titles_df["tag"].isin(description["tags"])].iterrows()
    ]

    titles = [msg[1]["content"] for msg in messages]

    print(f"{profile}: {len(titles)} titles to evaluate")

    scores = {}

    for title, message in zip(titles, messages):

        try:

            response = client.chat.completions.create(
                model=MODEL,
                messages=message,
                seed=SEED,
                temperature=TEMPERATURE,
            )

            content = response.choices[0].message.content

            json_match = re.search(r"```json\s*(.*?)\s*```", content, re.DOTALL)
            if json_match:
                json_str = json_match.group(1)
            else:
                json_str = content

            answer = json.loads(json_str)

        except Exception:

            answer = {"score": None}

        try:

            score = answer.get("score", None)
            score = int(score)
            scores[title] = score

        except (TypeError, ValueError):

            print(f"{profile}: something wrong went with title {title}")

    values = list(scores.values())
    if len(values):
        median = np.median(values)
    else:
        median = np.nan

    print(f"{profile}: {len(values)}, {median}")

    results[profile] = scores

  0%|          | 0/31 [00:00<?, ?it/s]

cultural_studies_enthusiast: 27 titles to evaluate


  3%|▎         | 1/31 [00:22<11:19, 22.66s/it]

cultural_studies_enthusiast: 27, 5.0
educational_content_developer: 160 titles to evaluate


  6%|▋         | 2/31 [02:42<44:10, 91.41s/it]

...

student_life_coordinator: 67, 5.0
urban_cultural_researcher: 33 titles to evaluate


 97%|█████████▋| 30/31 [41:27<01:03, 63.68s/it]

urban_cultural_researcher: 31, 5.0
mobile_app_developer: 24 titles to evaluate


100%|██████████| 31/31 [41:51<00:00, 81.02s/it]

mobile_app_developer: 19, 5.0


In [15]:
with open(f"{output_dir_name}/artificial_profiles_scores.pkl", "wb") as f:
    pickle.dump(results, f)